In [ ]:
import tkinter as tk
import cv2
import os
import numpy as np
from mtcnn import MTCNN
from keras_facenet import FaceNet
# import serial
import glob
import pickle

class MainGUI:
    def __init__(self, master):
        self.master = master
        master.title("Main Interface")

        # Create buttons
        self.registration_button = tk.Button(master, text="Registration", command=self.open_registration)
        self.registration_button.pack(padx=10, pady=10)

        self.capture_images_button = tk.Button(master, text="Capture Images", command=self.open_capture_images)
        self.capture_images_button.pack(padx=10, pady=10)

        self.train_images_button = tk.Button(master, text="Train Images", command=self.open_train_images)
        self.train_images_button.pack(padx=10, pady=10)

        self.start_door_automation_button = tk.Button(master, text="Start Door Automation", command=self.start_door_automation)
        self.start_door_automation_button.pack(padx=10, pady=10)

    def open_registration(self):
        registration_window = tk.Toplevel(self.master)
        registration_window.title("Registration Page")
        # Add registration form widgets here

    def open_capture_images(self):
        capture_images_window = tk.Toplevel(self.master)
        capture_images_window.title("Capture Images Page")

        # Create the directory if it doesn't exist
        if not os.path.exists('Captured Img'):
            os.makedirs('Captured Img')

        # Initialize MTCNN for face detection
        mtcnn = MTCNN()

        # Initialize the webcam
        cap = cv2.VideoCapture(0)

        # Set the counter for the image filename
        img_counter = 0

        while True:
            # Capture frame-by-frame
            ret, frame = cap.read()

            if not ret:
                print("Failed to capture image from webcam.")
                break
            
            # Detect faces in the frame using MTCNN
            faces = mtcnn.detect_faces(frame)

            # Draw bounding boxes around the detected faces
            for face in faces:
                x, y, w, h = face['box']
                cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)

            # Display the frame with bounding boxes
            cv2.imshow('Webcam', frame)

            # Press 'c' to capture an image
            if cv2.waitKey(1) & 0xFF == ord('c'):
                if len(faces) > 0:
                    img_name = f"Captured Img/img_{img_counter}.jpg"
                    cv2.imwrite(img_name, frame)
                    print(f"Image {img_counter} saved as {img_name}")
                    img_counter += 1

            # Press 'q' to quit
            elif cv2.waitKey(1) & 0xFF == ord('q'):
                break
            
        # Release the webcam and close all OpenCV windows
        cap.release()
        cv2.destroyAllWindows()

    def open_train_images(self):
        train_images_window = tk.Toplevel(self.master)
        train_images_window.title("Train Images Page")
        
        # Initialize MTCNN for face detection
        mtcnn = MTCNN()

        # Initialize the FaceNet model for face embeddings
        facenet = FaceNet()

        # Create a dictionary to store the face embeddings
        face_embeddings_dict = {}

        # Directory containing images for each class (person)
        image_directory = "images"  # Ensure this path is correct
        if not os.path.exists(image_directory):
            print(f"Directory {image_directory} does not exist.")
        else:
            for class_dir in glob.glob(os.path.join(image_directory, "*")):
                class_label = os.path.basename(class_dir)
                face_embeddings_dict[class_label] = []

                for img_path in glob.glob(os.path.join(class_dir, "*.jpg")):
                    # Load the face image
                    img = cv2.imread(img_path)
                    if img is None:
                        print(f"Could not load image: {img_path}")
                        continue

                    # Detect faces in the image using MTCNN
                    faces = mtcnn.detect_faces(img)
                    if len(faces) == 0:
                        print(f"No face detected in image: {img_path}")
                        continue

                    for i, face in enumerate(faces):
                        confidence = face['confidence']
                        x, y, w, h = face['box']
                        keypoints = face['keypoints']

                        print(f"Image: {img_path} | Confidence: {confidence:.2f} | Box: {x, y, w, h}")

                        x, y = max(0, x), max(0, y)
                        w, h = min(img.shape[1] - x, w), min(img.shape[0] - y, h)

                        face_img = img[y:y+h, x:x+w]

                        face_img = cv2.resize(face_img, (160, 160))

                        face_embedding = facenet.extract(face_img)

                        face_embeddings_dict[class_label].append(face_embedding)

                        cv2.rectangle(img, (x, y), (x+w, y+h), (0, 255, 0), 2)
                        cv2.putText(img, f"Conf: {confidence:.2f}", (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

                        for key, point in keypoints.items():
                            cv2.circle(img, point, 2, (0, 0, 255), 2)

                    cv2.imshow("Face Detection", img)
                    if cv2.waitKey(0) & 0xFF == ord('q'):
                        break

            with open("face_embeddings_dict.pkl", "wb") as f:
                pickle.dump(face_embeddings_dict, f)

            cv2.destroyAllWindows()

            print("Face embeddings successfully saved.")

    def start_door_automation(self):
        print("Door automation started!")

        # # Load the stored face embeddings from the pickle file
        with open("face_embeddings_dict.pkl", "rb") as f:
            face_embeddings_dict = pickle.load(f)

        # Initialize MTCNN for face detection
        import cv2
        import numpy as np
        import face_recognition as face_rec
        import os
        import pyttsx3 as text_speech
        from mtcnn import MTCNN
        import serial as ser
        import pandas as pd
        from datetime import datetime

        # Initialize text-to-speech engine
        engine = text_speech.init()

        # Initialize serial communication with Arduino
        # ser = ser.Serial('COM4', 9600, timeout=1)

        # Helper function to resize image
        def resize(img, size):
            width = int(img.shape[1] * size)
            height = int(img.shape[0] * size)
            dimension = (width, height)
            return cv2.resize(img, dimension, interpolation=cv2.INTER_AREA)

        # Path to user images for encoding comparison
        path = "user_images"
        user_images = []
        user_names = []

        # Load all user images and names
        myList = os.listdir(path)
        for user in myList:
            curr_image = cv2.imread(f'{path}/{user}')
            user_images.append(curr_image)
            user_names.append(os.path.splitext(user)[0])

        # Helper function to find encodings for the loaded images
        def find_encodings(images):
            encoding_list = []
            for img in images:
                img_resized = resize(img, 0.50)
                img_rgb = cv2.cvtColor(img_resized, cv2.COLOR_BGR2RGB)
                encode_img = face_rec.face_encodings(img_rgb)
                if encode_img:
                    encoding_list.append(encode_img[0])
            return encoding_list

        # Get the face encodings for all stored images
        encode_list = find_encodings(user_images)

        # Initialize MTCNN for face detection
        detector = MTCNN()

        # Start video capture from the webcam
        capture = cv2.VideoCapture(0)

        # Initialize DataFrames
        attendance_df = pd.DataFrame(columns=["Name", "Date", "Day", "Time"])
        attendance_log_df = pd.DataFrame(columns=["Name", "Date", "Day", "Time"])

        while True:
            success, frame = capture.read()
            if not success:
                print("Failed to capture frame from camera")
                continue
            
            # Resize frame for faster processing
            smaller_frame = cv2.resize(frame, (0, 0), fx=0.5, fy=0.5)
            rgb_small_frame = cv2.cvtColor(smaller_frame, cv2.COLOR_BGR2RGB)

            # Detect faces using MTCNN
            faces = detector.detect_faces(rgb_small_frame)

            for face in faces:
                x, y, width, height = face['box']
                face_location = (y, x + width, y + height, x)

                # Extract face encoding using face_recognition
                face_encoding = face_rec.face_encodings(rgb_small_frame, [face_location])

                if face_encoding:
                    face_encoding = face_encoding[0]
                    # Compare the encoding with known encodings
                    face_distances = face_rec.face_distance(encode_list, face_encoding)
                    best_match_index = np.argmin(face_distances)
                    if face_distances[best_match_index] < 0.7:  # Stricter threshold
                        name = user_names[best_match_index].upper()

                        # Get current time and date
                        now = datetime.now()
                        date_str = now.strftime("%Y-%m-%d")
                        day_str = now.strftime("%A")
                        time_str = now.strftime("%H:%M:%S")

                        # Log attendance if not already recorded today
                        if not ((attendance_df['Name'] == name) & (attendance_df['Date'] == date_str)).any():
                            new_row = pd.DataFrame([{"Name": name, "Date": date_str, "Day": day_str, "Time": time_str}])
                            attendance_df = pd.concat([attendance_df, new_row], ignore_index=True)

                        # Always log the pass-through instance
                        new_log_row = pd.DataFrame([{"Name": name, "Date": date_str, "Day": day_str, "Time": time_str}])
                        attendance_log_df = pd.concat([attendance_log_df, new_log_row], ignore_index=True)

                        # Save DataFrames to Excel files
                        attendance_df.to_excel("attendance.xlsx", index=False)
                        attendance_log_df.to_excel("attendance_log.xlsx", index=False)

                        # Send the open door signal to Arduino when a match is found
                        # ser.write(b'open_door\n')

                    else:
                        name = "Unknown"

                    # Scale back the face location to full size
                    top, right, bottom, left = [v * 2 for v in face_location]

                    # Draw a rectangle around the face
                    color = (0, 255, 0) if name != "Unknown" else (0, 0, 255)
                    cv2.rectangle(frame, (left, top), (right, bottom), color, 3)
                    cv2.rectangle(frame, (left, bottom - 35), (right, bottom), color, cv2.FILLED)
                    cv2.putText(frame, name, (left + 6, bottom - 6), cv2.FONT_HERSHEY_COMPLEX, 1, (255, 255, 255), 2)

                    # Text-to-speech
                    statement = f"The detected face is {name}" if name != "Unknown" else "No match found"
                    engine.say(statement)
                    engine.runAndWait()

            # Display the video feed with annotations
            cv2.imshow("Webcam Face Recognition", frame)

            # Break the loop if 'q' is pressed
            if cv2.waitKey(1) & 0xFF == ord('q'):
                
                break
            
        # Release resources
        capture.release()
        cv2.destroyAllWindows()
        ser.close()



root = tk.Tk()
app = MainGUI(root)
root.mainloop()


Door automation started!


In [ ]:
import tkinter as tk
import cv2
import os
import numpy as np
from mtcnn import MTCNN
from keras_facenet import FaceNet
import serial
import glob
import pickle


class MainGUI:
    def __init__(self, master):
        self.master = master
        master.title("Main Interface")

        # Create buttons
        self.registration_button = tk.Button(master, text="Registration", command=self.open_registration)
        self.registration_button.pack(padx=10, pady=10)

        self.capture_images_button = tk.Button(master, text="Capture Images", command=self.open_capture_images)
        self.capture_images_button.pack(padx=10, pady=10)

        self.train_images_button = tk.Button(master, text="Train Images", command=self.open_train_images)
        self.train_images_button.pack(padx=10, pady=10)

        self.start_door_automation_button = tk.Button(master, text="Start Door Automation", command=self.start_door_automation)
        self.start_door_automation_button.pack(padx=10, pady=10)

    def open_registration(self):
        registration_window = tk.Toplevel(self.master)
        registration_window.title("Registration Page")
        # Add registration form widgets here

    def open_capture_images(self):
        capture_images_window = tk.Toplevel(self.master)
        capture_images_window.title("Capture Images Page")

        # Create the directory if it doesn't exist
        if not os.path.exists('Captured Img'):
            os.makedirs('Captured Img')

        # Initialize MTCNN for face detection
        mtcnn = MTCNN()

        # Initialize the webcam
        cap = cv2.VideoCapture(0)

        # Set the counter for the image filename
        img_counter = 0

        while True:
            # Capture frame-by-frame
            ret, frame = cap.read()

            if not ret:
                print("Failed to capture image from webcam.")
                break
            
            # Detect faces in the frame using MTCNN
            faces = mtcnn.detect_faces(frame)

            # Draw bounding boxes around the detected faces
            for face in faces:
                x, y, w, h = face['box']
                cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)

            # Display the frame with bounding boxes
            cv2.imshow('Webcam', frame)

            # Press 'c' to capture an image
            if cv2.waitKey(1) & 0xFF == ord('c'):
                if len(faces) > 0:
                    img_name = f"Captured Img/img_{img_counter}.jpg"
                    cv2.imwrite(img_name, frame)
                    print(f"Image {img_counter} saved as {img_name}")
                    img_counter += 1

            # Press 'q' to quit
            elif cv2.waitKey(1) & 0xFF == ord('q'):
                break
            
        # Release the webcam and close all OpenCV windows
        cap.release()
        cv2.destroyAllWindows()

    def open_train_images(self):
        train_images_window = tk.Toplevel(self.master)
        train_images_window.title("Train Images Page")
        
        # Initialize MTCNN for face detection
        mtcnn = MTCNN()

        # Initialize the FaceNet model for face embeddings
        facenet = FaceNet()

        # Create a dictionary to store the face embeddings
        face_embeddings_dict = {}

        # Directory containing images for each class (person)
        image_directory = "images"  # Ensure this path is correct
        if not os.path.exists(image_directory):
            print(f"Directory {image_directory} does not exist.")
        else:
            for class_dir in glob.glob(os.path.join(image_directory, "*")):
                class_label = os.path.basename(class_dir)
                face_embeddings_dict[class_label] = []

                for img_path in glob.glob(os.path.join(class_dir, "*.jpg")):
                    # Load the face image
                    img = cv2.imread(img_path)
                    if img is None:
                        print(f"Could not load image: {img_path}")
                        continue

                    # Detect faces in the image using MTCNN
                    faces = mtcnn.detect_faces(img)
                    if len(faces) == 0:
                        print(f"No face detected in image: {img_path}")
                        continue

                    for i, face in enumerate(faces):
                        confidence = face['confidence']
                        x, y, w, h = face['box']
                        keypoints = face['keypoints']

                        print(f"Image: {img_path} | Confidence: {confidence:.2f} | Box: {x, y, w, h}")

                        x, y = max(0, x), max(0, y)
                        w, h = min(img.shape[1] - x, w), min(img.shape[0] - y, h)

                        face_img = img[y:y+h, x:x+w]

                        face_img = cv2.resize(face_img, (160, 160))

                        face_embedding = facenet.extract(face_img)

                        face_embeddings_dict[class_label].append(face_embedding)

                        cv2.rectangle(img, (x, y), (x+w, y+h), (0, 255, 0), 2)
                        cv2.putText(img, f"Conf: {confidence:.2f}", (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

                        for key, point in keypoints.items():
                            cv2.circle(img, point, 2, (0, 0, 255), 2)

                    cv2.imshow("Face Detection", img)
                    if cv2.waitKey(0) & 0xFF == ord('q'):
                        break

            with open("face_embeddings_dict.pkl", "wb") as f:
                pickle.dump(face_embeddings_dict, f)

            cv2.destroyAllWindows()

            print("Face embeddings successfully saved.")

    def start_door_automation(self):
        print("Door automation started!")

        # # Load the stored face embeddings from the pickle file
        with open("face_embeddings_dict.pkl", "rb") as f:
            face_embeddings_dict = pickle.load(f)

        # Initialize MTCNN for face detection
        import cv2
        import numpy as np
        import face_recognition as face_rec
        import os
        import pyttsx3 as text_speech
        from mtcnn import MTCNN
        import serial as ser
        import pandas as pd
        from datetime import datetime

        # Initialize text-to-speech engine
        engine = text_speech.init()

        # Initialize serial communication with Arduino
        ser = ser.Serial('COM4', 9600, timeout=1)

        # Helper function to resize image
        def resize(img, size):
            width = int(img.shape[1] * size)
            height = int(img.shape[0] * size)
            dimension = (width, height)
            return cv2.resize(img, dimension, interpolation=cv2.INTER_AREA)

        # Path to user images for encoding comparison
        path = "user_images"
        user_images = []
        user_names = []

        # Load all user images and names
        myList = os.listdir(path)
        for user in myList:
            curr_image = cv2.imread(f'{path}/{user}')
            user_images.append(curr_image)
            user_names.append(os.path.splitext(user)[0])

        # Helper function to find encodings for the loaded images
        def find_encodings(images):
            encoding_list = []
            for img in images:
                img_resized = resize(img, 0.50)
                img_rgb = cv2.cvtColor(img_resized, cv2.COLOR_BGR2RGB)
                encode_img = face_rec.face_encodings(img_rgb)
                if encode_img:
                    encoding_list.append(encode_img[0])
            return encoding_list

        # Get the face encodings for all stored images
        encode_list = find_encodings(user_images)

        # Initialize MTCNN for face detection
        detector = MTCNN()

        # Start video capture from the webcam
        capture = cv2.VideoCapture(0)

        # Initialize DataFrames
        attendance_df = pd.DataFrame(columns=["Name", "Date", "Day", "Time"])
        attendance_log_df = pd.DataFrame(columns=["Name", "Date", "Day", "Time"])

        while True:
            success, frame = capture.read()
            if not success:
                print("Failed to capture frame from camera")
                continue
            
            # Resize frame for faster processing
            smaller_frame = cv2.resize(frame, (0, 0), fx=0.5, fy=0.5)
            rgb_small_frame = cv2.cvtColor(smaller_frame, cv2.COLOR_BGR2RGB)

            # Detect faces using MTCNN
            faces = detector.detect_faces(rgb_small_frame)

            for face in faces:
                x, y, width, height = face['box']
                face_location = (y, x + width, y + height, x)

                # Extract face encoding using face_recognition
                face_encoding = face_rec.face_encodings(rgb_small_frame, [face_location])

                if face_encoding:
                    face_encoding = face_encoding[0]
                    # Compare the encoding with known encodings
                    face_distances = face_rec.face_distance(encode_list, face_encoding)
                    best_match_index = np.argmin(face_distances)
                    if face_distances[best_match_index] < 0.7:  # Stricter threshold
                        name = user_names[best_match_index].upper()

                        # Get current time and date
                        now = datetime.now()
                        date_str = now.strftime("%Y-%m-%d")
                        day_str = now.strftime("%A")
                        time_str = now.strftime("%H:%M:%S")

                        # Log attendance if not already recorded today
                        if not ((attendance_df['Name'] == name) & (attendance_df['Date'] == date_str)).any():
                            new_row = pd.DataFrame([{"Name": name, "Date": date_str, "Day": day_str, "Time": time_str}])
                            attendance_df = pd.concat([attendance_df, new_row], ignore_index=True)

                        # Always log the pass-through instance
                        new_log_row = pd.DataFrame([{"Name": name, "Date": date_str, "Day": day_str, "Time": time_str}])
                        attendance_log_df = pd.concat([attendance_log_df, new_log_row], ignore_index=True)

                        # Save DataFrames to Excel files
                        attendance_df.to_excel("attendance.xlsx", index=False)
                        attendance_log_df.to_excel("attendance_log.xlsx", index=False)

                        # Send the open door signal to Arduino when a match is found
                        ser.write(b'open_door\n')

                    else:
                        name = "Unknown"

                    # Scale back the face location to full size
                    top, right, bottom, left = [v * 2 for v in face_location]

                    # Draw a rectangle around the face
                    color = (0, 255, 0) if name != "Unknown" else (0, 0, 255)
                    cv2.rectangle(frame, (left, top), (right, bottom), color, 3)
                    cv2.rectangle(frame, (left, bottom - 35), (right, bottom), color, cv2.FILLED)
                    cv2.putText(frame, name, (left + 6, bottom - 6), cv2.FONT_HERSHEY_COMPLEX, 1, (255, 255, 255), 2)

                    # Text-to-speech
                    statement = f"The detected face is {name}" if name != "Unknown" else "No match found"
                    engine.say(statement)
                    engine.runAndWait()

            # Display the video feed with annotations
            cv2.imshow("Webcam Face Recognition", frame)

            # Break the loop if 'q' is pressed
            if cv2.waitKey(1) & 0xFF == ord('q'):
                
                break
            
        # Release resources
        capture.release()
        cv2.destroyAllWindows()
        ser.close()



root = tk.Tk()
app = MainGUI(root)
root.mainloop()


Image 0 saved as Captured Img/img_0.jpg
Image 1 saved as Captured Img/img_1.jpg
Image 2 saved as Captured Img/img_2.jpg
